Kết nối Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

Data Loader

In [7]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # PyTorch LSTM cần input 3 chiều: (batch_size, sequence_length, num_features)
        # Vì ta dùng lag features trên từng dòng, sequence_length = 1
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_dataloaders(data_path, train_path, val_path, test_path, batch_size=64):
    print("[INFO] Đang tải và xử lý dữ liệu...")

    # Đọc file tổng (xử lý dấu phẩy)
    df_full = pd.read_csv(data_path, thousands=',')
    df_full['time'] = pd.to_datetime(df_full['time'])
    df_full = df_full.sort_values(by='time').set_index('time')
    df_full = df_full[['load']]

    # Tạo features liền mạch
    df_full['dayofweek'] = df_full.index.dayofweek
    df_full['month'] = df_full.index.month
    df_full['lag_1'] = df_full['load'].shift(1)
    df_full['lag_7'] = df_full['load'].shift(7)
    df_full['rolling_mean_7'] = df_full['load'].shift(1).rolling(window=7).mean()

    # Đọc index từ 3 tập đã chia
    train_idx = pd.to_datetime(pd.read_csv(train_path)['time'])
    val_idx   = pd.to_datetime(pd.read_csv(val_path)['time'])
    test_idx  = pd.to_datetime(pd.read_csv(test_path)['time'])

    features = ['lag_1', 'lag_7', 'rolling_mean_7', 'dayofweek', 'month']
    target = 'load'

    # Map dữ liệu
    df_train = df_full.loc[train_idx].dropna()
    df_val   = df_full.loc[val_idx]
    df_test  = df_full.loc[test_idx]

    X_train, y_train = df_train[features].values, df_train[target].values
    X_val,   y_val   = df_val[features].values,   df_val[target].values
    X_test,  y_test  = df_test[features].values,  df_test[target].values

    # Chuẩn hóa (Scaling)
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train = scaler_X.fit_transform(X_train)
    X_val   = scaler_X.transform(X_val)
    X_test  = scaler_X.transform(X_test)

    y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_val   = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
    y_test  = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

    # Đóng gói DataLoader
    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    print("[INFO] Đã tạo xong DataLoader!")
    return train_loader, val_loader, test_loader, scaler_y

Mô hình kiến trúc LSTM (Mô hình mẫu chạy thử)

In [12]:
import torch.nn as nn

# ==========================================
# ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH LSTM
# ==========================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Lớp LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)

        # Lớp Fully Connected (Linear) để xuất ra giá trị dự báo
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        out, (hn, cn) = self.lstm(x)
        # Chỉ lấy output ở bước thời gian cuối cùng của sequence
        out = self.fc(out[:, -1, :])
        return out

EarlyStopping

In [14]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='best_model.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

Training Loop

In [9]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, patience, device, model_save_path):
    print(f"\n[INFO] Bắt đầu training trên thiết bị: {device}")
    model.to(device)
    early_stopping = EarlyStopping(patience=patience, verbose=True, path=model_save_path)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        # --- TRAIN ---
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y.view(-1, 1)) # Ép chiều (batch, 1)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        avg_train_loss = np.average(train_losses)
        history['train_loss'].append(avg_train_loss)

        # --- VALIDATION ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y.view(-1, 1))
                val_losses.append(loss.item())

        avg_val_loss = np.average(val_losses)
        history['val_loss'].append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

        # --- EARLY STOPPING ---
        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print("=> Đã kích hoạt Early Stopping! Ngừng training.")
            break

    print("=> Loading lại trọng số (weights) của best model...")
    model.load_state_dict(torch.load(model_save_path))
    return model, history

Training Model

In [15]:
DATA_PATH  = '/content/drive/MyDrive/clean_hourly.csv'
TRAIN_PATH = '/content/drive/MyDrive/train.csv'
VAL_PATH   = '/content/drive/MyDrive/val.csv'
TEST_PATH  = '/content/drive/MyDrive/test.csv'
MODEL_SAVE = '/content/drive/MyDrive/best_lstm.pth'

# 1. Load Data
train_loader, val_loader, test_loader, scaler_y = get_dataloaders(
    DATA_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH, batch_size=64
)

# 2. Khởi tạo Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# input_size = 5 do ta có 5 features: lag_1, lag_7, rolling_mean_7, dayofweek, month
model = LSTMModel(input_size=5, hidden_size=64, num_layers=2, output_size=1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Chạy Training
best_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=100,      # Chạy tối đa 100 epoch
    patience=10,         # Nếu 10 epoch liên tiếp val_loss không giảm thì dừng
    device=device,
    model_save_path=MODEL_SAVE
)

print("\n[INFO] Đã hoàn tất toàn bộ quy trình!")

[INFO] Đang tải và xử lý dữ liệu...
[INFO] Đã tạo xong DataLoader!

[INFO] Bắt đầu training trên thiết bị: cpu
Epoch [1/100] | Train Loss: 0.99940 | Val Loss: 0.00000
Validation loss decreased (inf --> 0.000002).  Saving model ...
Epoch [2/100] | Train Loss: 0.99744 | Val Loss: 0.00000
Validation loss decreased (0.000002 --> 0.000001).  Saving model ...
Epoch [3/100] | Train Loss: 0.99164 | Val Loss: 0.00001
EarlyStopping counter: 1 out of 10
Epoch [4/100] | Train Loss: 0.97125 | Val Loss: 0.00002
EarlyStopping counter: 2 out of 10
Epoch [5/100] | Train Loss: 0.93901 | Val Loss: 0.00001
EarlyStopping counter: 3 out of 10
Epoch [6/100] | Train Loss: 0.90018 | Val Loss: 0.00005
EarlyStopping counter: 4 out of 10
Epoch [7/100] | Train Loss: 0.87532 | Val Loss: 0.00002
EarlyStopping counter: 5 out of 10
Epoch [8/100] | Train Loss: 0.85646 | Val Loss: 0.00003
EarlyStopping counter: 6 out of 10
Epoch [9/100] | Train Loss: 0.84324 | Val Loss: 0.00002
EarlyStopping counter: 7 out of 10
Epoch [